# From Rigid Body to Affine Body Dynamics

This notebook shows how to construct the **Affine Body Dynamics (ABD) mass matrix**
from standard rigid body quantities: **mass** $m$, **center of mass** $\mathbf{c}$,
and **inertia tensor** $I$.

**Key result:** No mesh integration is needed — the ABD mass matrix can be built
purely from the rigid body properties that any physics engine or CAD tool provides.

References:
- [Affine Body Dynamics](affine_body_quantity.ipynb) — derives ABD quantities from mesh integration
- [Rigid IPC: compute_J](https://github.com/ipc-sim/rigid-ipc/blob/main/notebooks/compute_J.ipynb) — relates inertia to the $J$ (second moment) matrix

In [ ]:
import sympy as sp
from sympy import symbols, Matrix, eye, trace, Rational, simplify, zeros, integrate, diag
from sympy import init_printing
init_printing()

## 1. Rigid Body Dynamics Review

A rigid body in 3D is fully characterized by three quantities:

**Mass:**
$$m = \int_\Omega \rho \, dV$$

**Center of mass:**
$$\mathbf{c} = \frac{1}{m}\int_\Omega \rho \, \bar{\mathbf{x}} \, dV$$

**Inertia tensor** (about center of mass):
$$I^{cm}_{ij} = \int_\Omega \rho \left( |\bar{\mathbf{x}} - \mathbf{c}|^2 \delta_{ij} - (\bar{x}_i - c_i)(\bar{x}_j - c_j) \right) dV$$

The rigid body equations of motion are:
- Translation: $\mathbf{F} = m \ddot{\mathbf{c}}$
- Rotation (Euler's equation): $\boldsymbol{\tau} = I^{cm} \dot{\boldsymbol{\omega}} + \boldsymbol{\omega} \times (I^{cm} \boldsymbol{\omega})$

These three quantities $(m, \mathbf{c}, I^{cm})$ are what physics engines and CAD tools typically provide.

In [ ]:
m = symbols('m', positive=True)
c1, c2, c3 = symbols('c_1 c_2 c_3', real=True)
c = Matrix([c1, c2, c3])

Ixx, Iyy, Izz = symbols('I_{xx} I_{yy} I_{zz}')
Ixy, Ixz, Iyz = symbols('I_{xy} I_{xz} I_{yz}')
I_cm = Matrix([
    [Ixx, Ixy, Ixz],
    [Ixy, Iyy, Iyz],
    [Ixz, Iyz, Izz]
])

print("Rigid body inputs:")
print("mass =", m)
print("center of mass =")
display(c.T)
print("inertia tensor I^cm =")
I_cm

## 2. Affine Body Dynamics Mass Matrix

In ABD, each body has 12 DOFs: translation $\mathbf{t} \in \mathbb{R}^3$ and an affine matrix $A \in \mathbb{R}^{3\times 3}$.

A material point $\bar{\mathbf{x}}$ in the reference configuration maps to:
$$\mathbf{x} = \mathbf{t} + A\bar{\mathbf{x}} = J(\bar{\mathbf{x}})\,\mathbf{q}$$
where $\mathbf{q} = [\mathbf{t}^T,\, \mathrm{vec}(A)^T]^T \in \mathbb{R}^{12}$.

The kinetic energy gives the **mass matrix**:
$$T = \frac{1}{2}\dot{\mathbf{q}}^T M \dot{\mathbf{q}}, \quad M = \int_\Omega \rho \, J^T J \, dV$$

### Block Structure

Since $J = [I_3 \;\;|\;\; I_3 \otimes \bar{\mathbf{x}}^T]$, the product $J^TJ$ has entries:
- $M_{t_i, t_j} = m\,\delta_{ij}$
- $M_{t_i, A_{jk}} = \delta_{ij}\, m c_k$
- $M_{A_{ij}, A_{kl}} = \delta_{ik}\, S_{jl}$

where $S_{jl} = \int_\Omega \rho\, \bar{x}_j\, \bar{x}_l \, dV$ is the **second moment of mass** tensor.

In block form:
$$M = \begin{pmatrix} m\,I_3 & I_3 \otimes (m\mathbf{c}^T) \\ I_3 \otimes (m\mathbf{c}) & I_3 \otimes S \end{pmatrix}$$

So the ABD mass matrix depends on exactly three quantities:
- $m$ — total mass (scalar)
- $m\mathbf{c} = \int \rho\,\bar{\mathbf{x}}\,dV$ — first moment of mass ($3 \times 1$ vector)
- $S = \int \rho\,\bar{\mathbf{x}}\bar{\mathbf{x}}^T dV$ — second moment of mass ($3 \times 3$ symmetric matrix)

In [ ]:
xbar1, xbar2, xbar3 = symbols('xbar_1 xbar_2 xbar_3', real=True)
xbar = Matrix([xbar1, xbar2, xbar3])

J = Matrix.zeros(3, 12)
J[0:3, 0:3] = eye(3)
J[0, 3:6] = xbar.T
J[1, 6:9] = xbar.T
J[2, 9:12] = xbar.T

print("J(xbar) =")
display(J)

Sxx, Syy, Szz = symbols('S_{xx} S_{yy} S_{zz}')
Sxy, Sxz, Syz = symbols('S_{xy} S_{xz} S_{yz}')
S_sym = Matrix([
    [Sxx, Sxy, Sxz],
    [Sxy, Syy, Syz],
    [Sxz, Syz, Szz]
])

def build_abd_mass_matrix(mass, com, second_moment):
    """Build 12x12 ABD mass matrix from (m, c, S).

    M_{t_i, t_j} = m * delta_{ij}
    M_{t_i, A_{jk}} = delta_{ij} * m * c_k
    M_{A_{ij}, A_{kl}} = delta_{ik} * S_{jl}
    """
    M = zeros(12, 12)
    M[0:3, 0:3] = mass * eye(3)
    for i in range(3):
        for k in range(3):
            M[i, 3 + 3*i + k] = mass * com[k]
            M[3 + 3*i + k, i] = mass * com[k]
    for i in range(3):
        for j in range(3):
            for l in range(3):
                M[3 + 3*i + j, 3 + 3*i + l] = second_moment[j, l]
    return M

M_abd = build_abd_mass_matrix(m, c, S_sym)
print("ABD mass matrix M (12x12) =")
M_abd

## 3. From Rigid Body to Affine Body

The rigid body gives us $(m, \mathbf{c}, I^{cm})$. The ABD needs $(m, \mathbf{c}, S)$.

So the question reduces to: **can we recover $S$ from $I^{cm}$ and $(m, \mathbf{c})$?** Yes!

### Step 1: Relationship between $I$ and $S$

The inertia tensor about the **origin** is related to $S$ by definition:
$$I^O_{ij} = \int_\Omega \rho\,(|\bar{\mathbf{x}}|^2 \delta_{ij} - \bar{x}_i \bar{x}_j)\,dV = \mathrm{tr}(S)\,\delta_{ij} - S_{ij}$$

In matrix form: $\;I^O = \mathrm{tr}(S)\,I_3 - S$

### Step 2: Invert to get $S$ from $I^O$

Taking the trace: $\;\mathrm{tr}(I^O) = 3\,\mathrm{tr}(S) - \mathrm{tr}(S) = 2\,\mathrm{tr}(S)$

So $\mathrm{tr}(S) = \frac{1}{2}\mathrm{tr}(I^O)$, and therefore:

$$\boxed{S = \frac{1}{2}\,\mathrm{tr}(I^O)\,I_3 - I^O}$$

### Step 3: Get $I^O$ from $I^{cm}$ via the parallel axis theorem

$$\boxed{I^O = I^{cm} + m\left(|\mathbf{c}|^2 I_3 - \mathbf{c}\mathbf{c}^T\right)}$$

### Special case: diagonal inertia (principal axes, centered at origin)

When $\mathbf{c} = \mathbf{0}$ and $I^{cm} = \mathrm{diag}(I_x, I_y, I_z)$:

$$S = \mathrm{diag}\!\left(\frac{-I_x + I_y + I_z}{2},\; \frac{I_x - I_y + I_z}{2},\; \frac{I_x + I_y - I_z}{2}\right)$$

This is exactly the $J$ matrix from [rigid-ipc/compute_J.ipynb](https://github.com/ipc-sim/rigid-ipc/blob/main/notebooks/compute_J.ipynb).

In [ ]:
def inertia_to_second_moment(I_tensor):
    """Convert inertia tensor to second moment tensor.

    I_ij = tr(S) * delta_ij - S_ij
    =>  S = (1/2) * tr(I) * I_3 - I
    """
    return Rational(1, 2) * trace(I_tensor) * eye(3) - I_tensor

# Step 1: Parallel axis theorem — shift inertia from CoM to origin
I_origin = I_cm + m * (c.dot(c) * eye(3) - c * c.T)
print("I^O = I^cm + m*(|c|^2 * I_3 - c*c^T) =")
display(I_origin)

# Step 2: Second moment from I^O
S_from_rigid = simplify(inertia_to_second_moment(I_origin))
print("S = (1/2)*tr(I^O)*I_3 - I^O =")
display(S_from_rigid)

In [ ]:
# Verify roundtrip: S -> I^O -> S should be identity
s11, s22, s33, s12, s13, s23 = symbols('s11 s22 s33 s12 s13 s23')
S_test = Matrix([
    [s11, s12, s13],
    [s12, s22, s23],
    [s13, s23, s33]
])

I_from_S = trace(S_test) * eye(3) - S_test
S_roundtrip = inertia_to_second_moment(I_from_S)

print("Roundtrip verification S -> I^O -> S:")
print("S_recovered - S_original =", simplify(S_roundtrip - S_test))

# Special case: diagonal inertia, centered at origin
Ix, Iy, Iz = symbols('I_x I_y I_z', positive=True)
I_diag = diag(Ix, Iy, Iz)
S_diag = inertia_to_second_moment(I_diag)
print("\nDiagonal case (c=0, principal axes):")
print("S =")
S_diag

## 4. Numerical Verification: Uniform Cube

Let's verify the formula with a concrete example: a **uniform cube** of side $a$ with one corner at the origin (vertices from $\mathbf{0}$ to $(a, a, a)$).

Known rigid body quantities:
- $m = \rho a^3$
- $\mathbf{c} = (a/2, a/2, a/2)$
- $I^{cm} = \frac{ma^2}{6} I_3$ (diagonal, by symmetry of the cube)

We will:
1. Compute $S$ from the rigid body formula
2. Verify each entry of $S$ by direct integration $S_{jl} = \int_\Omega \rho\, x_j\, x_l\, dV$
3. Build the full $12\times 12$ ABD mass matrix and compare against direct integration of $\int \rho\, J^TJ\, dV$

In [ ]:
a = symbols('a', positive=True)
rho = symbols('rho', positive=True)

c_cube = Matrix([a/2, a/2, a/2])
I_cube_cm = Rational(1, 6) * m * a**2 * eye(3)

print("=== Cube (corner at origin, side a) ===")
print("c =", c_cube.T)
print("I^cm =")
display(I_cube_cm)

# Apply the rigid body -> ABD formula
I_O_cube = I_cube_cm + m * (c_cube.dot(c_cube) * eye(3) - c_cube * c_cube.T)
S_cube = simplify(inertia_to_second_moment(I_O_cube))

print("I^O =")
display(simplify(I_O_cube))
print("S =")
display(S_cube)

In [ ]:
# Verify S entries by direct integration over [0,a]^3
x, y, z = symbols('x y z')

S_xx_direct = integrate(rho * x**2, (x, 0, a), (y, 0, a), (z, 0, a)).subs(rho, m/a**3)
S_xy_direct = integrate(rho * x*y, (x, 0, a), (y, 0, a), (z, 0, a)).subs(rho, m/a**3)
mc_x_direct = integrate(rho * x, (x, 0, a), (y, 0, a), (z, 0, a)).subs(rho, m/a**3)

print("=== Direct integration verification ===")
print(f"m*c_x:  formula = {m * c_cube[0]},  direct = {simplify(mc_x_direct)},  match = {simplify(mc_x_direct - m * c_cube[0]) == 0}")
print(f"S_xx:   formula = {S_cube[0,0]},  direct = {simplify(S_xx_direct)},  match = {simplify(S_xx_direct - S_cube[0,0]) == 0}")
print(f"S_xy:   formula = {S_cube[0,1]},  direct = {simplify(S_xy_direct)},  match = {simplify(S_xy_direct - S_cube[0,1]) == 0}")

In [ ]:
# Full 12x12 mass matrix: rigid body formula vs direct integration
M_cube_rigid = build_abd_mass_matrix(m, c_cube, S_cube)

# Direct integration of M = integral(rho * J^T * J) over [0,a]^3
x1, x2, x3 = symbols('x1 x2 x3')
xb = Matrix([x1, x2, x3])
Jm = Matrix.zeros(3, 12)
Jm[0:3, 0:3] = eye(3)
Jm[0, 3:6] = xb.T
Jm[1, 6:9] = xb.T
Jm[2, 9:12] = xb.T

JTJ = Jm.T * Jm
M_direct = zeros(12, 12)
for i in range(12):
    for j in range(i, 12):
        if JTJ[i, j] != 0:
            val = integrate(rho * JTJ[i, j], (x1, 0, a), (x2, 0, a), (x3, 0, a))
            val = simplify(val.subs(rho, m/a**3))
            M_direct[i, j] = val
            M_direct[j, i] = val

diff = simplify(M_direct - M_cube_rigid)
print("=== Full 12x12 mass matrix comparison ===")
print("M_from_rigid_body - M_from_direct_integration =")
display(diff)
print()
if diff == zeros(12, 12):
    print("VERIFIED: ABD mass matrix from rigid body quantities matches direct integration!")
else:
    print("MISMATCH FOUND")

## 5. Summary: Recipe

Given rigid body quantities $(m, \mathbf{c}, I^{cm})$, build the $12\times 12$ ABD mass matrix as follows:

**Step 1 — Parallel axis theorem:** shift inertia to origin:
$$I^O = I^{cm} + m\left(|\mathbf{c}|^2 I_3 - \mathbf{c}\mathbf{c}^T\right)$$

**Step 2 — Inertia $\to$ second moment:** invert the relationship:
$$S = \frac{1}{2}\,\mathrm{tr}(I^O)\,I_3 - I^O$$

**Step 3 — Assemble** the $12\times 12$ mass matrix (entry-wise):
$$M_{t_i, t_j} = m\,\delta_{ij}, \quad M_{t_i, A_{jk}} = \delta_{ij}\,m\,c_k, \quad M_{A_{ij}, A_{kl}} = \delta_{ik}\,S_{jl}$$

Or in block form:
$$M = \begin{pmatrix} m\,I_3 & I_3 \otimes (m\mathbf{c}^T) \\ I_3 \otimes (m\mathbf{c}) & I_3 \otimes S \end{pmatrix}$$

**No mesh integration is needed** — only standard rigid body properties.